# System archetypes in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/11-archetypes.ipynb)

This notebook is the Python companion to the [System archetypes](https://waelrash1.github.io/loopbricks/#/learn/archetypes) module. An archetype is a feedback story that keeps turning up in very different places. Here you run four of them as real models: **limits to growth**, **shifting the burden**, **tragedy of the commons** and **eroding goals**. For each one you see the behaviour the story predicts, then test the leverage point the story suggests.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Limits to growth: the engine is not the problem

A reinforcing loop makes a population grow, and the same growth uses up the room it needs. In **Constrained Growth** (`logistic`), births are a fixed fraction of the population, and crowding deaths rise as the population fills its carrying capacity. Early on the reinforcing loop wins; near the limit the balancing loop takes over.

In [ ]:
lg = lb.molecule("logistic")
print(lg.equations())
df = lg.run()
net = df["Growth"] - df["Crowding deaths"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
df["Population"].plot(ax=ax1, title="Population"); ax1.axhline(200, ls=":", c="grey")
net.plot(ax=ax2, title="Net growth per year")
for ax in (ax1, ax2): ax.grid(alpha=0.3)
print(f"Net growth is fastest in year {net.idxmax():.1f}, when the population is {df['Population'][net.idxmax()]:.0f}")

The curve is an S. Net growth peaks when the population is about half the carrying capacity of 200, which is the moment the balancing loop becomes as strong as the reinforcing one. After that, every extra member makes growth slower, not faster.

**Try it.** The archetype says that pushing harder on the growth engine does not help for long. Compare doubling the growth rate (`growth_rate=35`) with raising the limit (`carrying_capacity=350`). Which one changes where the population ends up?

<details><summary>Show a worked answer</summary>

The next cell runs both. With a growth rate of 35 the population reaches 190 in about 17 years instead of 33, but it still stops at 200. Raising the carrying capacity to 350 is the only change that moves the plateau. The leverage is in the limit, not in the engine.
</details>

In [ ]:
for label, kw in [("base", {}), ("faster engine", {"growth_rate": 35}), ("higher limit", {"carrying_capacity": 350})]:
    p = lg.run(**kw)["Population"]
    print(f"{label:<14} reaches 190 in year {p.index[p >= 190][0]:>5.1f}, ends at {p.iloc[-1]:.0f}")

## 2. Shifting the burden: the quick fix eats the cure

A team has two ways to deal with problems. A **quick fix** patches the problem, but patches fail after a while and the problem comes back. A **fundamental fix** removes it for good, but it needs **capability**, and capability is built only with spare time. In **Shifting the Burden** (`shiftingBurden`), the `Share patched quickly` sets how much the team leans on patches. Patching uses team capacity, which leaves less spare time to build capability.

In [ ]:
sb = lb.molecule("shiftingBurden")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
for preset in ("Invest in the fix", "Balanced", "Firefighting"):
    d = sb.run(preset=preset)
    d["Open problems"].plot(ax=ax1, label=preset)
    d["Capability"].plot(ax=ax2, label=preset)
ax1.set_title("Open problems"); ax2.set_title("Capability"); ax1.legend()
for ax in (ax1, ax2): ax.grid(alpha=0.3)

ff = sb.run(preset="Firefighting")["Open problems"]
print(f"Firefighting: open problems fall to {ff.min():.0f} in week {ff.idxmin():.1f}, "
      f"pass the starting 73 again in week {ff.index[(ff.index > ff.idxmin()) & (ff > 73)][0]:.0f}, "
      f"and end at {ff.iloc[-1]:.0f}")

Firefighting looks like the best policy for the first few weeks: open problems drop from 73 to about 59. Then the failed patches come back, capability decays because nobody has time to build it, and by week 60 there are more than three times as many open problems. Investing in the fundamental fix looks worse at first and ends best. That gap between the short run and the long run is why the trap is so easy to fall into.

**Try it.** Run 200 weeks and sweep `share_patched_quickly` from 0.1 to 0.9. Is the damage gradual, or is there a tipping point?

<details><summary>Show a worked answer</summary>

There is a tipping point between 0.5 and 0.6. Up to 0.5, more patching costs a little capability each time (66.8 at 0.1, 53.1 at 0.5). At 0.6, quick fixes use up the whole team capacity of 80, spare time falls to zero, capability collapses to about 0.5 and open problems run past 1,000. Once the quick fix takes all the time, the fundamental fix stops for good.
</details>

In [ ]:
for share in (0.1, 0.3, 0.5, 0.6, 0.8):
    d = sb.run(T=200, share_patched_quickly=share)
    print(f"share {share}: open problems {d['Open problems'].iloc[-1]:7.1f}, capability {d['Capability'].iloc[-1]:5.1f}")

## 3. Tragedy of the commons: each fleet is right, together they are wrong

Two fleets fish the same stock. Fish regrow fastest when the stock is half its carrying capacity. Each fleet adds boats while a boat earns more than its break-even catch, and scraps boats when it earns less. No one decides to wreck the fishery: each fleet simply responds to its own profits. **Tragedy of the Commons** (`tragedyCommons`) runs this story, and a `Catch limit` above zero caps the total catch.

In [ ]:
tc = lb.molecule("tragedyCommons")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
for preset in ("Open access", "Catch limit of 8"):
    d = tc.run(preset=preset)
    d["Fish stock"].plot(ax=ax1, label=preset)
    d["Catch"].plot(ax=ax2, label=preset)
ax1.set_title("Fish stock"); ax2.set_title("Catch per year"); ax1.legend()
for ax in (ax1, ax2): ax.grid(alpha=0.3)

oa = tc.run(preset="Open access")
print(oa.loc[[0, 2, 4, 8, 20, 60], ["Fish stock", "Fleet A", "Fleet B", "Catch", "Catch per boat"]].round(2))

With open access, both fleets grow fast while catches are good, the catch peaks at about 34 a year in year 2.6, and the stock crashes to about 7 by year 8. Boats are scrapped, the stock recovers a little, and by year 60 the fish stock is near 26, where a boat's catch (0.26) is only just above break-even (0.25). Open access drives the stock to the level where fishing stops being profitable, not to the level where fish regrow fastest. A catch limit of 8 keeps the stock above 70.

**Try it.** The fish regrow at most `0.4 × 50 × (1 − 50/100) = 10` a year. Sweep the catch limit from 6 to 12. Which limit lands the most fish over 60 years, and what happens just above it?

<details><summary>Show a worked answer</summary>

A limit of 10 lands the most, 600 over 60 years, with the stock drifting down towards 50. A limit of 11 asks for more than the stock can ever regrow, so the stock keeps falling (to about 23) and the fleets cannot catch even the limit: the total drops to 567. The best rule sits right at the regrowth peak, and going a little past it is much worse than staying a little below it.
</details>

In [ ]:
for limit in (6, 8, 10, 11, 12):
    d = tc.run(catch_limit=limit)   # Catch is a rate per year; times dt = 0.1 and summed, it is the total catch
    print(f"limit {limit:>2}: total catch {d['Catch'].sum() * 0.1:5.0f}, fish stock at year 60 {d['Fish stock'].iloc[-1]:5.1f}")

## 4. Eroding goals: a target that follows performance

When performance falls short of a goal, you can improve performance or lower the goal. In **Sea Anchor & Adjustment** (`seaAnchor`) the `Anchor` is the goal, and performance (`Produced value`) is the goal times an `Adjustment pressure`. A pressure of 0.9 means the team always delivers 10% less than the goal. The goal then adjusts towards what was delivered, with an `Anchor lag` of 6 months.

**Protected Sea Anchor & Adjustment** (`protectedSeaAnchor`) adds the leverage point from the archetype: the goal is partly tied to a fixed external standard, the `Fundamental` of 100. `Protection` is the weight on that standard.

In [ ]:
sa = lb.molecule("seaAnchor")
fig, ax = plt.subplots(figsize=(8, 4))
for pressure in (1, 0.95, 0.9):
    d = sa.run(adjustment_pressure=pressure)
    d["Anchor"].plot(ax=ax, label=f"delivers {pressure:.0%} of the goal")
    print(f"pressure {pressure}: goal after 12 months {d['Anchor'][12]:5.1f}, after 60 months {d['Anchor'].iloc[-1]:5.1f}")
ax.set_ylabel("goal (Anchor)"); ax.legend(); ax.grid(alpha=0.3)

ps = lb.molecule("protectedSeaAnchor")
for protection in (0, 0.25, 0.5, 0.9):
    d = ps.run(adjustment_pressure=0.9, protection=protection)
    print(f"protected, weight {protection:<4}: goal at month 60 {d['Anchor'].iloc[-1]:5.1f}, performance {d['Value'].iloc[-1]:5.1f}")

A 10% shortfall looks harmless in any one month: after a year the goal has only slipped to about 82. But the slide never stops, because each new goal is set from the last disappointing result. After five years the goal is 37, and performance follows it down.

Protection changes the picture. With no weight on the external standard the goal erodes exactly as before. Even a weight of 0.25 stops the slide: the goal levels off near 77. With a weight of 0.5 it settles at 90.9, which is `0.5 × 100 / (1 − 0.5 × 0.9)`: the point where the pull of the standard balances the pull of poor results.

**Try it.** Someone suggests a different fix: review the goal less often, by raising the `anchor_lag` from 6 to 12 months (with `adjustment_pressure=0.9` and no protection). Does that stop the erosion?

<details><summary>Show a worked answer</summary>

It only slows it. With a lag of 12 months the goal is 60.6 after 60 months instead of 36.8, and it is still falling at the same 10% shortfall per lag. A slower goal buys time; only an anchor outside the system's own results gives the goal a floor.
</details>

In [ ]:
for lag in (3, 6, 12):
    d = sa.run(adjustment_pressure=0.9, anchor_lag=lag)
    print(f"anchor lag {lag:>2} months: goal after 60 months {d['Anchor'].iloc[-1]:5.1f}")

## Where next

- Back in the app: the [System archetypes](https://waelrash1.github.io/loopbricks/#/learn/archetypes) module has the causal loop diagrams and stories for all eleven archetypes, including fixes that fail, escalation and success to the successful, each with the molecules that carry it.
- The S-curve from section 1 is the subject of [S-shaped growth](https://waelrash1.github.io/loopbricks/#/learn/s-shaped); goals that drift and anchors that adjust come back in [Bounded rationality and expectations](https://waelrash1.github.io/loopbricks/#/learn/behaviour).
- Every archetype here is a small model. Try writing one yourself with `lb.Model.from_spec`, as in the [stocks and flows notebook](02-stocks-flows.ipynb), for example a fix that fails because overtime builds fatigue.
- For the stories behind these structures, see Senge (1990) and Sterman (2000, ch. 4 and 5).